In [1]:
import pandas as pd
import unicodedata
import re
from pathlib import Path

base_data_dir = Path("./data")
input_files = sorted(base_data_dir.glob("*/Evaluación-Table 1.csv"))
output_file = base_data_dir / "legal-governance-financing-analysis.csv"

def norm(s):
    s = "" if s is None else str(s)
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

rename_map = {
    "Código": "action_id",
    "Acción / Sub-criterios": "action_name",
    "Puntaje\nTotal": "final_score",
    "Puntaje\nLegal\n(40%)": "legal_score_40",
    "Titularidad\ncompetencia\n60%": "legal_ownership_60",
    "Restricciones\njurídicas\n30%": "legal_restrictions_30",
    "Alineación\npolítica\nnacional\n10%": "national_policy_alignment_10",
    "Justificación Legal": "legal_justification",
    "Puntaje\nGob.\n(30%)": "governance_score_30",
    "Capacidad\ntécnica e\ninstitucional\n35%": "technical_institutional_capacity_35",
    "Dependencia\nactores\nexternos\n30%": "external_dependency_30",
    "Instrumento\nlocal\nhabilitante\n25%": "local_enabling_instrument_25",
    "Precedente\nen otras\ncomunas\n10%": "precedent_other_municipalities_10",
    "Justificación Gobernanza": "governance_justification",
    "Puntaje\nFin.\n(30%)": "financing_score_30",
    "Disponibilidad\nfuente\nidentificada\n75%": "funding_source_availability_75",
    "Accesibilidad\ndel\nfinanciamiento\n25%": "funding_accessibility_25",
    "Justificación Financiamiento": "financing_justification",
    "1 Norma / Fuente  →  click para abrir": "legal_reference_1",
    "2 Norma / Fuente  →  click para abrir": "legal_reference_2",
    "3 Norma / Fuente  →  click para abrir": "legal_reference_3",
    "4 Norma / Fuente  →  click para abrir": "legal_reference_4",
    "5 Norma / Fuente  →  click para abrir": "legal_reference_5",
    "6 Norma / Fuente  →  click para abrir": "legal_reference_6",
}

score_cols = [
    "final_score","legal_score_40","legal_ownership_60","legal_restrictions_30",
    "national_policy_alignment_10","governance_score_30","technical_institutional_capacity_35",
    "external_dependency_30","local_enabling_instrument_25","precedent_other_municipalities_10",
    "financing_score_30","funding_source_availability_75","funding_accessibility_25"
]

all_dfs = []

for input_csv in input_files:
    raw = pd.read_csv(input_csv, header=None, dtype=str, keep_default_na=False)
    header_idx = next((i for i in range(len(raw)) if norm(raw.iloc[i, 0]) == "codigo"), None)
    if header_idx is None:
        continue

    header = raw.iloc[header_idx].tolist()
    df = raw.iloc[header_idx + 1:].copy()
    df.columns = header
    df = df.loc[:, [c for c in df.columns if norm(c) != ""]]
    if "Código" in df.columns:
        df = df[df["Código"].astype(str).str.strip().ne("PESOS →")]

    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    folder = input_csv.parent.name.lower()
    if "transporte" in folder:
        sector = "transport"
    elif "residuos" in folder:
        sector = "waste"
    elif "energia_tramo1" in folder:
        sector = "energy_tramo1"
    elif "energia_tramo2" in folder:
        sector = "energy_tramo2"
    else:
        sector = folder

    df["sector"] = sector
    df["source_folder"] = input_csv.parent.name
    df["source_file"] = input_csv.name

    for c in score_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    all_dfs.append(df)

legal_analysis = pd.concat(all_dfs, ignore_index=True)
legal_analysis.insert(0, "id", range(1, len(legal_analysis) + 1))
legal_analysis.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved {output_file} with shape {legal_analysis.shape}")

Saved data/legal-governance-financing-analysis.csv with shape (49, 28)


In [4]:
import json
import pandas as pd
import numpy as np

# Input / output
in_path = "data/legal-governance-financing-analysis.csv"
actions_path = "data/actions.json"
out_path = "data/legal-analysis-v1.csv"

df = pd.read_csv(in_path)

# Ensure numeric
df["legal_ownership_60"] = pd.to_numeric(df["legal_ownership_60"], errors="coerce")
df["legal_restrictions_30"] = pd.to_numeric(df["legal_restrictions_30"], errors="coerce")

# Normalized scores (0..1)
df["ownership_score"] = df["legal_ownership_60"] / 100.0
df["restrictions_score"] = df["legal_restrictions_30"] / 100.0

# Weights + weighted overall
df["ownership_weight"] = 0.67
df["restrictions_weight"] = 0.33
df["verdict_score"] = (
    df["ownership_weight"] * df["ownership_score"]
    + df["restrictions_weight"] * df["restrictions_score"]
).round(2)

# Category helpers
def score_to_category(v):
    if pd.isna(v):
        return None
    if v == 100:
        return "enabled"
    if v == 50:
        return "conditional"
    if v == 0:
        return "blocked"
    return "conditional"  # safe fallback for non-standard scores

df["ownership_category"] = df["legal_ownership_60"].apply(score_to_category)
df["restrictions_category"] = df["legal_restrictions_30"].apply(score_to_category)

# Overall verdict
df["verdict_category"] = np.select(
    [
        (df["legal_ownership_60"] == 0) | (df["legal_restrictions_30"] == 0),
        (df["legal_ownership_60"] == 100) & (df["legal_restrictions_30"] == 100),
    ],
    [
        "blocked",
        "enabled",
    ],
    default="conditional",
)

# Descriptions from methodology maps
ownership_desc_map = {
    100: "Municipality has explicit legal authority to act directly.",
    50: "Authority exists but is conditional, ambiguous, or mediated by an enabling instrument.",
    0: "Authority belongs to another level of government; municipality cannot act alone.",
}
restrictions_desc_map = {
    100: "No legal restrictions; no additional authorization required.",
    50: "Moderate legal risk; may require prior authorization or face potential legal challenge.",
    0: "There is a legal prohibition/restriction, or legal reform is needed.",
}

df["ownership_description"] = df["legal_ownership_60"].round().map(ownership_desc_map)
df["restrictions_description"] = df["legal_restrictions_30"].round().map(restrictions_desc_map)

# Optional fallback for non-standard values
df["ownership_description"] = df["ownership_description"].fillna("Non-standard score value.")
df["restrictions_description"] = df["restrictions_description"].fillna("Non-standard score value.")

# Keep output columns (column-wise structure, not row-wise criteria)
out_cols = [
    "action_id",
    "action_name",
    "sector",
    "verdict_category",
    "verdict_score",
    "ownership_score",
    "ownership_category",
    "ownership_weight",
    "ownership_description",
    "restrictions_score",
    "restrictions_category",
    "restrictions_weight",
    "restrictions_description",
    "legal_justification",
    "legal_reference_1",
    "legal_reference_2",
    "legal_reference_3",
    "legal_reference_4",
    "legal_reference_5",
    "legal_reference_6",
]

# Keep only columns that exist in current file
out_cols = [c for c in out_cols if c in df.columns]
clean = df[out_cols].copy()

# SQL actions CTE equivalent
actions = pd.read_json(actions_path)
actions = actions[actions["ActionType"].apply(lambda x: isinstance(x, list) and "mitigation" in x)].copy()
actions = actions.explode("Sector", ignore_index=True)
actions["sector"] = actions["Sector"].astype(str).str.replace("Transport", "transportation", regex=False)
actions["action_name_en"] = actions["ActionName"].apply(lambda x: x.get("en") if isinstance(x, dict) else None)
actions["action_name_es"] = actions["ActionName"].apply(lambda x: x.get("es") if isinstance(x, dict) else None)
actions = actions[["ActionID", "action_name_en", "action_name_es", "sector"]].rename(columns={"ActionID": "action_id"})

# SQL all_data CTE equivalent
all_data = actions.merge(clean, on="action_id", how="left", suffixes=("", "_lac"))
if "sector_lac" in all_data.columns:
    all_data = all_data.drop(columns=["sector_lac"])

# Final select equivalent
final_cols = [
    "action_id",
    "action_name_en",
    "action_name_es",
    "sector",
    "verdict_category",
    "verdict_score",
    "ownership_category",
    "ownership_score",
    "ownership_weight",
    "ownership_description",
    "restrictions_category",
    "restrictions_score",
    "restrictions_weight",
    "restrictions_description",
    "legal_justification",
    "legal_reference_1",
    "legal_reference_2",
    "legal_reference_3",
    "legal_reference_4",
    "legal_reference_5",
    "legal_reference_6",
]
final = all_data[final_cols].copy()
final = final[final["action_id"] != "icare_0002"].copy()

# Bilingual legal justification payload for downstream API mocks
final["legal_justification_iln8"] = final["legal_justification"].apply(
    lambda x: json.dumps({"es": x, "en": ""}, ensure_ascii=False)
)

final["analysis_date"] = "2026-04-30"
final["generation_method"] = "expert review"
final["publisher_id"] = "SSG"

ordered_cols = [
    "action_id",
    "action_name_en",
    "action_name_es",
    "sector",
    "verdict_category",
    "verdict_score",
    "ownership_category",
    "ownership_score",
    "ownership_weight",
    "ownership_description",
    "restrictions_category",
    "restrictions_score",
    "restrictions_weight",
    "restrictions_description",
    "legal_justification",
    "analysis_date",
    "generation_method",
    "publisher_id",
    "legal_reference_1",
    "legal_reference_2",
    "legal_reference_3",
    "legal_reference_4",
    "legal_reference_5",
    "legal_reference_6",
]

final = final[ordered_cols].sort_values(["sector", "action_id"], ascending=[False, True])

# Export final output
final.to_csv(out_path, index=False)
print(f"Saved final: {out_path} rows={len(final)}")
final.head()

Saved final: data/legal-analysis-v1.csv rows=102


,action_id,action_name_en,action_name_es,sector,verdict_category,verdict_score,ownership_category,ownership_score,ownership_weight,ownership_description,...,legal_justification,analysis_date,generation_method,publisher_id,legal_reference_1,legal_reference_2,legal_reference_3,legal_reference_4,legal_reference_5,legal_reference_6
10,c40_0034,Enact and enforce material bans in order to re...,Promulgar y hacer cumplir prohibiciones de mat...,waste,conditional,0.50,conditional,0.5,0.67,"Authority exists but is conditional, ambiguous...",...,"La Ley 21.100, arts. 1 y 3, prohíbe a los esta...",2026-04-30,expert review,SSG,Ley 21.100 — Bolsas plásticas — BCN,Ley 18.695 (LOCM) — BCN,Sentencia R-521-2025 — Segundo Tribunal Ambiental,Expediente R-521-2025 — Tribunal Ambiental,NaN,NaN
11,c40_0035,Optmize waste management systems,Optimizar los sistemas de gestión de residuos,waste,conditional,0.84,enabled,1.0,0.67,Municipality has explicit legal authority to a...,...,La Ley 18.695 art. 3 d) establece el aseo y or...,2026-04-30,expert review,SSG,Ley 18.695 (LOCM) — BCN,Guía Metodológica Planes Residuos MMA,FNDR — GORE (referencia general),NaN,NaN,NaN
12,c40_0036,Upgrade Landfills to Engineered Sanitary Landf...,Actualizar los vertederos a rellenos sanitario...,waste,blocked,0.00,blocked,0.0,0.67,Authority belongs to another level of governme...,...,Los rellenos sanitarios son instalaciones de d...,2026-04-30,expert review,SSG,D.S. 189/2005 Ministerio de Salud — BCN,Ley 18.695 (LOCM) — BCN,Ley 19.300 Bases del Medio Ambiente (SEIA) — BCN,D.S. 136/2004 Reglamento Orgánico MINSAL — BCN,NaN,NaN
13,c40_0037,Implement residential and business segregated ...,Implementar la recolección segregada de residu...,waste,blocked,0.34,conditional,0.5,0.67,"Authority exists but is conditional, ambiguous...",...,La Ley 18.695 art. 3 d) establece un mandato m...,2026-04-30,expert review,SSG,Ley 18.695 (LOCM) — BCN,Ley 20.920 REP — BCN,DS 12/2020 — Metas envases y embalajes — BCN,Res. Ex. N°9/2022 MMA — Modelo Ordenanza REP,DS 7/2017 MMA — Fondo para el Reciclaje,NaN
14,c40_0040,Implement Volume-Based Collection policy,Implementar la política de recolección basada ...,waste,conditional,0.50,conditional,0.5,0.67,"Authority exists but is conditional, ambiguous...",...,"El DL N° 3.063 de 1979 (Rentas Municipales), a...",2026-04-30,expert review,SSG,DL 3.063 Rentas Municipales — BCN,Ley 18.695 (LOCM) — BCN,Dictamen CGR N°29.860/2006 — vLex,Dictamen CGR N°33.416/2016 — Contraloría,Dictamen CGR N°4.059/2016 — vLex,NaN


In [7]:
import csv
import json
from datetime import datetime, timezone
from pathlib import Path

csv_path = Path("data/legal-analysis-v1.csv")
out_path = Path("data/actions_legal_api_mock.json")
country_code = "CL"

with csv_path.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))


def _to_float(v, default=None):
    try:
        return float(v) if v not in (None, "") else default
    except ValueError:
        return default


def _non_empty(values):
    return [v.strip() for v in values if v and v.strip()]


actionsLegal = []
for r in rows:
    actionId = (r.get("action_id") or "").strip()
    if not actionId:
        continue

    legalReferences = _non_empty(
        [r.get(f"legal_reference_{i}", "") for i in range(1, 7)]
    )

    actionsLegal.append(
        {
            "actionId": actionId,
            "verdict": {
                "category": (r.get("verdict_category") or "").strip() or None,
                "score": _to_float(r.get("verdict_score")),
            },
            "signals": {
                "ownership": {
                    "category": (r.get("ownership_category") or "").strip() or None,
                    "score": _to_float(r.get("ownership_score")),
                    "weight": _to_float(r.get("ownership_weight")),
                },
                "restrictions": {
                    "category": (r.get("restrictions_category") or "").strip() or None,
                    "score": _to_float(r.get("restrictions_score")),
                    "weight": _to_float(r.get("restrictions_weight")),
                },
            },
            "legalEvidence": {
                "legalJustification": (r.get("legal_justification") or "").strip() or None,
                "legalReferences": legalReferences,
            },
            "metadata": {
                "countryCode": country_code,
                "analysisDate": (r.get("analysis_date") or "").strip() or None,
                "generationMethod": (r.get("generation_method") or "").strip() or None,
                "publisherId": (r.get("publisher_id") or "").strip() or None,
            },
        }
    )

payload = {
    "meta": {
        "generatedAtUtc": datetime.now(timezone.utc).isoformat(),
        "endpoint": "GET /v1/actions/city/{locode}/legal",
        "countryCode": country_code,
        "totalRecords": len(actionsLegal),
    },
    "actionsLegal": actionsLegal,
}

out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {out_path} with {len(actionsLegal)} records")

Wrote data/actions_legal_api_mock.json with 102 records
